# Stackformer: Sequence-to-Sequence Transformer Training

This tutorial demonstrates configuring and training an encoder-decoder Transformer on a sequence-to-sequence task using `Trainer`.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset

from stackformer.models.language import Transformer
from stackformer.training import TrainConfig, Trainer

# 1. Model Definition: Vaswani-style Encoder-Decoder Transformer
model = Transformer(
    vocab_size=300,
    embed_dim=64,
    num_heads=2,
    encoder_layers=2,
    decoder_layers=2,
    seq_len=32,
    hidden_dim=128,
)
print(f"Transformer parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Prepare Seq2Seq Dataset

For sequence-to-sequence models with distinct `source` and `target` arguments, batches are yielded as dictionary mappings matching `Transformer.forward(source, target)` parameter names.

In [ ]:
class Seq2SeqDataset(Dataset):
    def __init__(self, size=32, seq_len=16, vocab_size=300):
        torch.manual_seed(42)
        self.src = torch.randint(1, vocab_size, (size, seq_len))
        self.tgt = torch.randint(1, vocab_size, (size, seq_len))

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return {
            "source": self.src[idx],
            "target": self.tgt[idx],
            "labels": self.tgt[idx],
        }


train_loader = DataLoader(Seq2SeqDataset(32, 16, 300), batch_size=8, shuffle=True)
val_loader = DataLoader(Seq2SeqDataset(16, 16, 300), batch_size=8)

## 3. Train with Stackformer Trainer

In [ ]:
cfg = TrainConfig(
    max_epochs=2,
    lr=1e-3,
    loss="cross_entropy",
    checkpoint_dir="./checkpoints_seq2seq",
    eval_every_n_epochs=1,
    display="auto",
)

trainer = Trainer(model, cfg, train_loader=train_loader, eval_loader=val_loader)
state = trainer.train()
print(f"Seq2Seq training completed! Global step: {state.global_step}")